In [1]:
# =============================================================================
# IMPORT LIBRARIES
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import pickle
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

In [2]:
# =============================================================================
# PROJECT PATHS
# =============================================================================

PROJECT_ROOT = Path("../")

DATA_DIR = PROJECT_ROOT / "Data"
PROCESSED_DIR = DATA_DIR / "Processed"
MODEL_DIR = PROJECT_ROOT / "models"

print(PROCESSED_DIR)
print(MODEL_DIR)

../Data/Processed
../models


In [3]:
# =============================================================================
# LOAD DATA
# =============================================================================

df = pd.read_parquet(
    PROCESSED_DIR / "engineered_features.parquet"
)

print(df.shape)

df.head()

(3429120, 53)


,timestamp,state,city,latitude,longitude,calculated_aqi,calculated_aqi_lag_1,calculated_aqi_rolling_std_3,pm10,pm25,...,nitric_oxide,solar_radiation,nitrogen_oxides,sulfur_dioxide_ambient_temperature,nitrogen_dioxide_solar_radiation,dayofweek,DayOfWeek_sin,rainfall,DayOfWeek_cos,is_weekend
0,2022-10-21 12:00:00,Chhattisgarh,"32Bungalows, Bhilai (, )",21.194815,81.31477,90.105553,0.0,-0.464454,-0.051401,0.137457,...,-0.520691,0.033731,-0.670562,1.412033,-0.130775,0.25,-0.277479,0.0,-0.445042,0.0
1,2022-10-21 13:00:00,Chhattisgarh,"32Bungalows, Bhilai (, )",21.194815,81.31477,90.105553,0.0,-0.464454,-0.051401,0.137457,...,-0.520691,0.033731,-0.670562,1.412033,-0.130775,0.25,-0.277479,0.0,-0.445042,0.0
2,2022-10-21 14:00:00,Chhattisgarh,"32Bungalows, Bhilai (, )",21.194815,81.31477,90.105553,0.0,-0.464454,-0.051401,0.137457,...,-0.520691,0.033731,-0.670562,1.412033,-0.130775,0.25,-0.277479,0.0,-0.445042,0.0
3,2022-10-21 15:00:00,Chhattisgarh,"32Bungalows, Bhilai (, )",21.194815,81.31477,90.105553,0.0,-0.464454,-0.051401,0.137457,...,-0.520691,0.033731,-0.670562,1.412033,-0.130775,0.25,-0.277479,0.0,-0.445042,0.0
4,2022-10-21 16:00:00,Chhattisgarh,"32Bungalows, Bhilai (, )",21.194815,81.31477,90.105553,0.0,-0.464454,-0.059219,0.109711,...,-0.513358,-0.024683,-0.664270,1.354955,-0.152182,0.25,-0.277479,0.0,-0.445042,0.0


In [4]:
# =============================================================================
# LOAD SELECTED FEATURES
# =============================================================================

with open(MODEL_DIR / "selected_features.pkl", "rb") as f:
    selected_features = pickle.load(f)

print("Number of Features:", len(selected_features))

Number of Features: 47


In [5]:
# =============================================================================
# PREPARE TEST DATA
# =============================================================================

TARGET = "calculated_aqi"

X = df[selected_features]
y = df[TARGET]

split_index = int(len(df) * 0.8)

X_test = X.iloc[split_index:].copy()
y_test = y.iloc[split_index:].copy()

print(X_test.shape)

(685824, 47)


In [6]:
# =============================================================================
# LOAD TRAINED MODELS
# =============================================================================

cat_model = CatBoostRegressor()
cat_model.load_model(
    MODEL_DIR / "catboost_model.cbm"
)

lgb_model = joblib.load(
    MODEL_DIR / "lightgbm_model.pkl"
)

xgb_model = xgb.Booster()
xgb_model.load_model(
    str(MODEL_DIR / "xgboost_model.json")
)

print("Models Loaded Successfully")

Models Loaded Successfully


In [7]:
# =============================================================================
# GENERATE PREDICTIONS
# =============================================================================

cat_pred = cat_model.predict(X_test)

lgb_pred = lgb_model.predict(X_test)

xgb_pred = xgb_model.predict(
    xgb.DMatrix(X_test)
)

print(cat_pred.shape)

(685824,)


In [8]:
# =============================================================================
# INDIVIDUAL MODEL PERFORMANCE
# =============================================================================

models = {
    "CatBoost": cat_pred,
    "LightGBM": lgb_pred,
    "XGBoost": xgb_pred
}

for name, pred in models.items():

    mae = mean_absolute_error(y_test, pred)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)

    print(f"{name:12s}  MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.6f}")

CatBoost      MAE=0.9205  RMSE=2.6745  R2=0.998600
LightGBM      MAE=1.0073  RMSE=2.7710  R2=0.998497
XGBoost       MAE=1.0403  RMSE=2.7766  R2=0.998491


In [9]:
# =============================================================================
# WEIGHTED ENSEMBLE
# =============================================================================

weights = {
    "CatBoost":0.40,
    "LightGBM":0.30,
    "XGBoost":0.30
}

ensemble_pred = (
      weights["CatBoost"] * cat_pred
    + weights["LightGBM"] * lgb_pred
    + weights["XGBoost"] * xgb_pred
)

In [10]:
# =============================================================================
# ENSEMBLE EVALUATION
# =============================================================================

ensemble_mae = mean_absolute_error(
    y_test,
    ensemble_pred
)

ensemble_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        ensemble_pred
    )
)

ensemble_r2 = r2_score(
    y_test,
    ensemble_pred
)

print("="*70)

print("ENSEMBLE RESULTS")

print("="*70)

print(f"MAE  : {ensemble_mae:.4f}")
print(f"RMSE : {ensemble_rmse:.4f}")
print(f"R2   : {ensemble_r2:.6f}")

ENSEMBLE RESULTS
MAE  : 0.8999
RMSE : 2.6477
R2   : 0.998628


In [11]:
# =============================================================================
# COMPARISON TABLE
# =============================================================================

comparison = pd.DataFrame({

    "Model":[
        "CatBoost",
        "LightGBM",
        "XGBoost",
        "Weighted Ensemble"
    ],

    "MAE":[
        mean_absolute_error(y_test,cat_pred),
        mean_absolute_error(y_test,lgb_pred),
        mean_absolute_error(y_test,xgb_pred),
        ensemble_mae
    ],

    "RMSE":[
        np.sqrt(mean_squared_error(y_test,cat_pred)),
        np.sqrt(mean_squared_error(y_test,lgb_pred)),
        np.sqrt(mean_squared_error(y_test,xgb_pred)),
        ensemble_rmse
    ],

    "R2":[
        r2_score(y_test,cat_pred),
        r2_score(y_test,lgb_pred),
        r2_score(y_test,xgb_pred),
        ensemble_r2
    ]
})

comparison.sort_values(
    "RMSE"
)

,Model,MAE,RMSE,R2
3,Weighted Ensemble,0.899922,2.647678,0.998628
0,CatBoost,0.920513,2.674522,0.998600
1,LightGBM,1.007289,2.770958,0.998497
2,XGBoost,1.040314,2.776561,0.998491


In [12]:
# =============================================================================
# SAVE ENSEMBLE INFORMATION
# =============================================================================

ensemble_info = {

    "weights":weights,

    "features":selected_features,

    "metrics":{

        "MAE":ensemble_mae,
        "RMSE":ensemble_rmse,
        "R2":ensemble_r2

    }

}

with open(
    MODEL_DIR/"ensemble_info.pkl",
    "wb"
) as f:

    pickle.dump(
        ensemble_info,
        f
    )

print("Ensemble information saved.")

Ensemble information saved.


In [13]:
# =============================================================================
# SAVE ENSEMBLE PREDICTIONS
# =============================================================================

prediction_df = pd.DataFrame({

    "Actual":y_test,

    "CatBoost":cat_pred,

    "LightGBM":lgb_pred,

    "XGBoost":xgb_pred,

    "Ensemble":ensemble_pred

})

prediction_df.head(10)

,Actual,CatBoost,LightGBM,XGBoost,Ensemble
2743296,43.756248,43.357350,43.690512,43.096359,43.379002
2743297,46.102085,45.130876,44.538382,44.879089,44.877592
2743298,47.570835,47.669785,47.092744,47.134418,47.336064
2743299,46.156250,45.471790,45.575002,45.318542,45.456780
2743300,45.149307,44.582179,44.712461,44.830555,44.695777
2743301,44.181713,43.444835,43.550204,43.003689,43.344102
2743302,43.253471,43.157827,43.032917,42.847706,43.027318
2743303,42.251736,41.894728,41.871568,41.408962,41.742051
2743304,41.381367,41.027850,41.150311,40.822853,41.003090
2743305,40.642361,40.370904,40.491819,39.954048,40.282122
